# 02 · Limpieza de datos — ModaModerna

In [357]:
# Cargar librerías

# Librerías nativas de Python
import os

# Manipulación y análisis de datos
import numpy as np
import pandas as pd

# Gráficos y visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar opciones para que Pandas no corte las columnas ni la pantalla
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Tema global de gráficos
sns.set_theme(style='whitegrid', palette='colorblind')

In [358]:
# Cargar los archivos

clientes = pd.read_csv('datos/clientes.csv')          
productos = pd.read_csv('datos/productos.csv')        
campana = pd.read_csv('datos/campana.csv')            
pedidos = pd.read_csv('datos/pedidos.csv')            
lineas_pedido = pd.read_csv('datos/lineas_pedido.csv')
devoluciones = pd.read_csv('datos/devoluciones.csv')  
visitas_web = pd.read_csv('datos/visitas_web.csv')    

# Tabla clientes: Limpieza

In [359]:
# Observaciones de exploración a corregir:
# 1. cliente_id: pasar a tipo texto (str)
# 2. fecha_registro: conversión a datetime 
# 3. genero: estandarización de variantes a 2 categorías (M/F)
# 4. ciudad: estandarización de mayúsculas/minúsculas
# 5. edad: valores fuera de rango de negocio (16-90) pasan a NaN ( así no se ven afectadas las estadísticas)

clientes_limpio = clientes.copy()

# 1. Clave primaria: conversión a tipo texto
clientes_limpio['cliente_id'] = clientes_limpio['cliente_id'].astype(str)

In [360]:
# 2. Fechas: conversión a datetime

# Usamos pd.to_datetime() para convertir la columna a tipo datetime,
# ya que está diseñada específicamente para trabajar con fechas (mejor que astype).

# La columna mezcla dos formatos (yyyy-mm-dd y dd/mm/yyyy), así que debemos
# usar pd.to_datetime() con el parámetro 'format' para cada formato por separado.
# Sin usar el formato correto, pd.to_datetime() puede interpretar mal las fechas y generar errores.

fechas_formato_ano_mes_dia = pd.to_datetime(clientes_limpio['fecha_registro'], format='%Y-%m-%d', errors='coerce')
fechas_formato_dia_mes_ano = pd.to_datetime(clientes_limpio['fecha_registro'], format='%d/%m/%Y', errors='coerce')

# al aplicar pd.to_datetime(), el formato resultante para los dos casos es año-mes-día (ISO), que es el estándar internacional y el recomendado para trabajar con fechas en Python.

# errors='coerce' convierte en NaT los valores que no se pueden interpretar
# como fechas, evitando que el código genere un error.

In [361]:
# 3. Género: estandarización de variantes a 2 categorías (M/F)
# .str.strip() → elimina espacios al principio y al final.
# .str.upper() → convierte a mayúsculas.
# .replace() → reemplaza los valores según el diccionario proporcionado. 

clientes_limpio['genero'] = clientes_limpio['genero'].str.strip().str.upper().replace({
    'MASCULINO': 'M',
    'FEMENINO': 'F',
})

# Comprobamos si el diccionario de géneros cubre todos los valores únicos de la columna 'genero' después de la limpieza. 
# Si no, podemos identificar qué valores faltan y agregarlos al diccionario.
sorted(clientes_limpio['genero'].unique())

['F', 'M']

In [362]:
# 4. Normalización de ciudades

# .str → permite aplicar métodos de texto a la columna.
# .str.strip() → elimina espacios al principio y al final.
# .str.lower() → convierte los textos a minúsculas.
# .replace(ciudades) → reemplaza los valores según el diccionario proporcionado. 
#
# No usamos .map() porque si no seleccionamos todos los valores, 
# los valores que no estén en el diccionario
# se convertirían en NaN. Con .replace() evitamos perder esos valores.

# El diccionario recoge como primer valor la ciudad que aparece en los datos
# y como segundo valor la ciudad que queremos obtener después de la limpieza.
# cada pareja tiene un valor de origen y uno de destino.

ciudades = {
    "barcelona": "Barcelona",
    "l'hospitalet de llobregat": "L'Hospitalet de Llobregat",
    "badalona": "Badalona",
    "terrassa": "Terrassa",
    "sabadell": "Sabadell",
    "mataró": "Mataró",
    "girona": "Girona",
    "tarragona": "Tarragona",
    "lleida": "Lleida",
    "reus": "Reus",
    "manresa": "Manresa",
    "sant cugat del vallès": "Sant Cugat del Vallès",
    "vic": "Vic",
}

clientes_limpio['ciudad'] = (
    clientes_limpio['ciudad']
    .str.strip()
    .str.lower()
    .replace(ciudades)
)


# Comprobamos si el diccionario de ciudades cubre todos los valores únicos de la columna 'ciudad' después de la limpieza. 
# Si no, podemos identificar qué valores faltan y agregarlos al diccionario.
print(sorted(clientes_limpio['ciudad'].unique()))

['Badalona', 'Barcelona', 'Girona', "L'Hospitalet de Llobregat", 'Lleida', 'Manresa', 'Mataró', 'Reus', 'Sabadell', 'Sant Cugat del Vallès', 'Tarragona', 'Terrassa', 'Vic']


In [363]:
# 5. Edad: valores fuera del rango de negocio (16-90 años) se consideran errores y se pasan a NaN
# clientes_limpio['edad'][condición] = np.nan determina que según la condición, los valores de la columna 'edad' se reemplazarán por NaN. 
# Esto es útil para limpiar datos que no cumplen con los criterios de negocio sin eliminar toda la fila, preservando así la información válida de otros campos del cliente.
clientes_limpio['edad'][(clientes_limpio['edad'] < 16) | (clientes_limpio['edad'] > 90)] = np.nan

# Resultados
print('Muestra de clientes limpios:', clientes_limpio.head(10))

Muestra de clientes limpios:   cliente_id fecha_registro                     ciudad genero  edad
0     100000     27/02/2024                  Barcelona      M  34.0
1     100001     2024-10-28                       Reus      M  32.0
2     100002     2024-08-24                     Lleida      F  38.0
3     100003     2023-10-23  L'Hospitalet de Llobregat      M  21.0
4     100004     01/09/2022                  Barcelona      M  35.0
5     100005     2024-12-06                  Barcelona      F  43.0
6     100006     2025-01-04  L'Hospitalet de Llobregat      M  35.0
7     100007     2021-08-04                       Reus      F  54.0
8     100008     10/07/2023                  Barcelona      F  19.0
9     100009     2023-12-07  L'Hospitalet de Llobregat      F  28.0


C:\Users\USER\AppData\Local\Temp\ipykernel_2856\726325057.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  clientes_limpio['edad'][(clientes_limpio['edad'] < 16) | (clientes_limpio['edad'] > 90)] = np.nan
C:\Users\USER\AppData\Local\Temp\

In [364]:
os.makedirs('datos_limpios', exist_ok=True)
clientes_limpio.to_csv('datos_limpios/clientes_limpio.csv', index=False)

# Tabla productos: Limpieza

In [365]:
# Observaciones de exploración a corregir:
# 1. producto_id: tipo correcto (str)
# 2. categoria: estandarización de mayúsculas/minúsculas y singular/plural
# 3. temporada: estandarización de variantes de escritura
# 4. coste_unitario, precio_base: ya están en formato numérico correcto, no requieren limpieza

productos_limpio = productos.copy()

# 1. Conversión del identificador de producto a tipo texto
productos_limpio['producto_id'] = productos_limpio['producto_id'].astype(str)

In [366]:
# 2. Categoría: mismas categorías escritas de formas distintas (mayúsculas/minúsculas, singular/plural)

# El diccionario recoge como primer valor la categoría que aparece en los datos
# y como segundo valor la categoría que queremos obtener después de la limpieza.
# cada pareja tiene un valor de origen y uno de destino.

categoria = {
    'vestidos': 'Vestidos', 'vestido': 'Vestidos',
    'camisetas': 'Camisetas', 'camiseta': 'Camisetas',
    'pantalones': 'Pantalones', 'pantalon': 'Pantalones',
    'faldas': 'Faldas', 'falda': 'Faldas',
    'jerseis': 'Jerseis', 'jersey': 'Jerseis', 'jerséis': 'Jerseis',
    'abrigos': 'Abrigos', 'abrigo': 'Abrigos',
    'calzado': 'Calzado',
    'zapatos': 'Zapatos',
    'accesorios': 'Accesorios', 'accesorio': 'Accesorios',
    'complementos': 'Complementos',
}
productos_limpio['categoria'] = productos_limpio['categoria'].str.strip().str.lower().replace(categoria)

# Comprobamos si el diccionario de categorías cubre todos los valores únicos de la columna categoria después de la limpieza. 
# Si no, podemos identificar qué valores faltan y agregarlos al diccionario.
print(sorted(productos_limpio['categoria'].unique()))

['Abrigos', 'Accesorios', 'Calzado', 'Camisetas', 'Complementos', 'Faldas', 'Jerseis', 'Pantalones', 'Vestidos', 'Zapatos']


In [367]:
# 3. Temporada: misma temporada escrita de formas distintas ( abreviaturas)

# El diccionario recoge como primer valor la temporada que aparece en los datos
# y como segundo valor la temporada que queremos obtener después de la limpieza.
# cada pareja tiene un valor de origen y uno de destino.

temporada = {
    'verano 2025': 'Primavera/Verano 2025',
    'verano': 'Primavera/Verano 2025',
    'verano25': 'Primavera/Verano 2025',
    'ss25': 'Primavera/Verano 2025',
    'otoño invierno 2025': 'Otoño/Invierno 2025',
    'invierno 2025': 'Otoño/Invierno 2025',
    'invierno25': 'Otoño/Invierno 2025',
    'oi25': 'Otoño/Invierno 2025',
}
productos_limpio['temporada'] = productos_limpio['temporada'].str.strip().str.lower().replace(temporada)

# Comprobamos si el diccionario de temporadas cubre todos los valores únicos de la columna 'temporada' después de la limpieza. 
# Si no, podemos identificar qué valores faltan y agregarlos al diccionario.
print(sorted(productos_limpio['temporada'].unique()))

# Resultados
print('Muestra de productos limpios:', productos_limpio.head(10))

['Otoño/Invierno 2025', 'Primavera/Verano 2025']
Muestra de productos limpios:   producto_id          nombre   categoria              temporada         coleccion  coste_unitario  precio_base
0        1001  Sandalia Petra     Zapatos    Otoño/Invierno 2025      Mediterráneo            9.67        24.95
1        1002   Sandalia Flor     Zapatos  Primavera/Verano 2025  Fiesta Diciembre           20.46        52.00
2        1003      Zapato Sol     Calzado  Primavera/Verano 2025             Noche           27.63        51.14
3        1004       Bolso Ona  Accesorios    Otoño/Invierno 2025       Cápsula Sol           43.86       110.29
4        1005   Sandalia Iris     Zapatos  Primavera/Verano 2025        Playa Club           12.02        28.20
5        1006    Sandalia Ona     Zapatos  Primavera/Verano 2025  Fiesta Diciembre           24.31        47.44
6        1007   Vestido Petra    Vestidos  Primavera/Verano 2025        Urban SS25           23.05        56.84
7        1008     Zapato 

In [368]:
os.makedirs('datos_limpios', exist_ok=True)
productos_limpio.to_csv('datos_limpios/productos_limpio.csv', index=False)

# Tabla campaña: Limpieza

In [369]:
# Observaciones de exploración a corregir:
# 1. campana_id: tipo correcto (str)
# 2. fecha_inicio, fecha_fin: tipo correcto (datetime)
# 3. presupuesto: símbolo de moneda y separador de miles en una sola fila

campana_limpio = campana.copy()

# 1. Conversión de columna id a tipo str
campana_limpio['campana_id'] = campana_limpio['campana_id'].astype(str)

# 2. Conversión de columnas de fecha a formato datetime (mismo formato para todas las filas)
campana_limpio['fecha_inicio'] = pd.to_datetime(campana_limpio['fecha_inicio'], format='%Y-%m-%d')
campana_limpio['fecha_fin'] = pd.to_datetime(campana_limpio['fecha_fin'], format='%Y-%m-%d')

In [370]:
# 3. Presupuesto:
# La columna está almacenada como texto (object) y presenta diferentes formatos:
# números sin formato moneda (ej. 68000.0) y números con formato moneda (ej. 64.000,00 €).
# En análisis de datos el formato sin moneda es el que necesitamos para poder realizar el análisis posterior.
# Podemos aplicar a la columna presupuesto la transformación para eliminar el símbolo de euro, el separador de miles y cambiar la coma decimal por un punto decimal.
# Después convertimos toda la columna a tipo numérico (float) con to_numeric.

campana_limpio['presupuesto'] = (
    campana_limpio['presupuesto']
        .str.replace('€', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
)

campana_limpio['presupuesto'] = pd.to_numeric(
    campana_limpio['presupuesto'],
    errors='coerce'
)

# Resultados
print('Muestra presupuesto:', campana_limpio['presupuesto'])

Muestra presupuesto: 0    680000
1     64000
2    820000
3    760000
Name: presupuesto, dtype: int64


In [371]:
os.makedirs('datos_limpios', exist_ok=True)
campana_limpio.to_csv('datos_limpios/campana_limpio.csv', index=False)

# Tabla pedidos: Limpieza

In [372]:
# Observaciones de exploración a corregir:
# 1. pedido_id, cliente_id, campana_id: tipo correcto (str), campana_id(float)
# 2. fecha_pedido: conversión a datetime (formato mixto)
# 3. ciudad: estandarización de mayúsculas/minúsculas 

pedidos_limpio = pedidos.copy()

# 1. IDs: Conversión de identificadores a tipo texto (string)

pedidos_limpio['pedido_id'] = pedidos_limpio['pedido_id'].astype(str)
pedidos_limpio['cliente_id'] = pedidos_limpio['cliente_id'].astype(str)

# 'campana_id' es un identificador almacenado como float.
# Se convierte primero a entero para eliminar el '.0' y después a texto.
# Si lo pasáramos directamente a string, los valores conservarían el '.0'.
# La lógica es: float → Int64 → string.
pedidos_limpio['campana_id'] = (
    pedidos_limpio['campana_id']
        .astype('Int64')
        .astype('string')
)

In [373]:
# 2. Fechas: mismo criterio que en clientes (dos formatos)
fechas_ano_mes_dia = pd.to_datetime(pedidos_limpio['fecha_pedido'], format='%Y-%m-%d', errors='coerce')
fechas_dia_mes_ano = pd.to_datetime(pedidos_limpio['fecha_pedido'], format='%d/%m/%Y', errors='coerce')

In [374]:
# 3. Ciudad: 
ciudades = {
    'vic': 'Vic',
    'barcelona': 'Barcelona',
    'sant cugat del vallès': 'Sant Cugat del Vallès',
    'tarragona': 'Tarragona',
    "l'hospitalet de llobregat": "L'Hospitalet de Llobregat",
    'girona': 'Girona',
    'lleida': 'Lleida',
    'manresa': 'Manresa',
    'sabadell': 'Sabadell',
    'badalona': 'Badalona',
    'terrassa': 'Terrassa',
    'reus': 'Reus',
    'mataró': 'Mataró'
}

pedidos_limpio['ciudad'] = pedidos_limpio['ciudad'].str.strip().str.lower().replace(ciudades)

# Comprobamos si el diccionario de ciudades cubre todos los valores únicos de la columna 'ciudad' después de la limpieza. 
# Si no, podemos identificar qué valores faltan y agregarlos al diccionario.
print(sorted(pedidos_limpio['ciudad'].unique()))


# Resultados
print('Muestra de pedidos limpios:', pedidos_limpio.head(10))

['Badalona', 'Barcelona', 'Girona', "L'Hospitalet de Llobregat", 'Lleida', 'Manresa', 'Mataró', 'Reus', 'Sabadell', 'Sant Cugat del Vallès', 'Tarragona', 'Terrassa', 'Vic']
Muestra de pedidos limpios:   pedido_id cliente_id fecha_pedido campana_id      canal                     ciudad metodo_pago
0    500001     106480   14/07/2025          2  instagram                        Vic     tarjeta
1    500002     123135   2025-11-13       <NA>     google                  Barcelona      paypal
2    500003     127034   2025-09-04       <NA>    directo      Sant Cugat del Vallès       bizum
3    500004     130452   2025-11-03       <NA>     google                  Tarragona     tarjeta
4    500005     109360   2025-01-17          1     google  L'Hospitalet de Llobregat      paypal
5    500006     139183   14/12/2025          4  instagram                  Barcelona     tarjeta
6    500007     133599   2025-11-11       <NA>      email                  Barcelona     tarjeta
7    500008     106204 

In [375]:
os.makedirs('datos_limpios', exist_ok=True)
pedidos_limpio.to_csv('datos_limpios/pedidos_limpio.csv', index=False)

# Tabla líneas de pedido: Limpieza

In [376]:
# Observaciones de exploración a corregir:
# 1. linea_id, pedido_id, producto_id: tipo correcto (str)
# 2. precio_unitario: símbolo de moneda y coma decimal mezclados con número plano
# 3. descuento_pct: normalizar a una única fracción decimal (0-1)

lineas_pedido_limpio = lineas_pedido.copy()

# 1. IDs: Conversión de identificadores a tipo texto (string)
lineas_pedido_limpio['linea_id'] = lineas_pedido_limpio['linea_id'].astype(str)
lineas_pedido_limpio['pedido_id'] = lineas_pedido_limpio['pedido_id'].astype(str)
lineas_pedido_limpio['producto_id'] = lineas_pedido_limpio['producto_id'].astype(str)

In [377]:
# 2. precio_unitario: quitar símbolo de moneda y pasar la coma decimal a punto
lineas_pedido_limpio['precio_unitario'] = pd.to_numeric(
    lineas_pedido_limpio['precio_unitario'].astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip(),
    errors='coerce'
)

In [378]:
# 3. descuento_pct: normalización a fracción decimal (0-1)

# Paso 1: La columna está almacenada como texto (string).
# Eliminamos el símbolo "%" de los valores que lo contienen.
lineas_pedido_limpio['descuento_pct'] = (
    lineas_pedido_limpio['descuento_pct']
    .astype(str)
    .str.replace('%', '', regex=False)
)

# Paso 2: Convertimos la columna de texto a número (float).
lineas_pedido_limpio['descuento_pct'] = (
    lineas_pedido_limpio['descuento_pct'].astype(float)
)

# Paso 3: mask sirve para crear una condición que identifica qué filas cumplen algo
# buscamos los valores mayores que 1, que corresponden
# a descuentos expresados como porcentaje, 
# y los dividimos entre 100. Los valores que ya
# están entre 0 y 1 (0.1, 0.2, 0.3, etc.) no cambian.

lineas_pedido_limpio['descuento_pct'] = (
    lineas_pedido_limpio['descuento_pct']
    .mask(
        lineas_pedido_limpio['descuento_pct'] > 1,
        lineas_pedido_limpio['descuento_pct'] / 100
    )
)

# Resultados
print('Muestra de las líneas de pedido limpias:', lineas_pedido_limpio.head(10))

Muestra de las líneas de pedido limpias:   linea_id pedido_id producto_id talla  cantidad  precio_unitario  descuento_pct
0   900001    500001        1567    XS         1            20.86            0.5
1   900002    500002        1463     S         1            66.70            0.0
2   900003    500002        1313     M         1            35.42            0.0
3   900004    500002        1565    XS         1            44.57            0.0
4   900005    500003        1393     S         1            53.11            0.0
5   900006    500003        1385     L         2            76.44            0.1
6   900007    500004        1486    36         1            47.93            0.0
7   900008    500005        1035     L         1            24.26            0.0
8   900009    500005        1117    42         2            31.93            0.5
9   900010    500005        1369    XL         1            90.16            0.3


In [379]:
os.makedirs('datos_limpios', exist_ok=True)
lineas_pedido_limpio.to_csv('datos_limpios/lineas_pedido_limpio.csv', index=False)

# Tabla devoluciones: Limpieza

In [380]:
# Observaciones de exploración a corregir:
# 1. devolucion_id, linea_id: tipo correcto (str)
# 2. fecha_devolucion: conversión a datetime (formato mixto)
# 3. talla: estandarización de mayúsculas/minúsculas

devoluciones_limpio = devoluciones.copy()

devoluciones_limpio['devolucion_id'] = devoluciones_limpio['devolucion_id'].astype(str)
devoluciones_limpio['linea_id'] = devoluciones_limpio['linea_id'].astype(str)

fechas_iso = pd.to_datetime(devoluciones_limpio['fecha_devolucion'], format='%Y-%m-%d', errors='coerce')
fechas_europeas = pd.to_datetime(devoluciones_limpio['fecha_devolucion'], format='%d/%m/%Y', errors='coerce')
devoluciones_limpio['fecha_devolucion'] = fechas_iso.fillna(fechas_europeas)

devoluciones_limpio['talla'] = devoluciones_limpio['talla'].str.upper()

# Resultados
print('Muestra de devoluciones limpias:', devoluciones_limpio.head(10))

Muestra de devoluciones limpias:   devolucion_id linea_id fecha_devolucion talla            motivo
0        700001  1009435       2025-12-16     S  talla incorrecta
1        700002   927467       2025-03-24    42           defecto
2        700003   994165       2026-01-02     L  talla incorrecta
3        700004   913307       2025-08-30     M  no me queda bien
4        700005   956270       2025-11-27     S           defecto
5        700006   947894       2025-12-02     M    color distinto
6        700007  1009202       2025-09-11     S  talla incorrecta
7        700008   936140       2025-12-22     S  talla incorrecta
8        700009   948474       2026-01-15     S       no me gusta
9        700010  1074122       2025-05-18     S  no me queda bien


In [381]:
os.makedirs('datos_limpios', exist_ok=True)
devoluciones_limpio.to_csv('datos_limpios/devoluciones_limpio.csv', index=False)

# Tabla visitas web: Limpieza

In [382]:
# Observaciones de exploración a corregir:
# 1. fecha: conversión a datetime (formato ya consistente, yyyy-mm-dd, sin mezcla)

visitas_web_limpio = visitas_web.copy()

# fechas con mismo formato, pero convertimos a datetime para poder trabajar con ellas en análisis posteriores.
visitas_web_limpio['fecha'] = pd.to_datetime(visitas_web_limpio['fecha'], format='%Y-%m-%d', errors='coerce')

In [383]:
os.makedirs('datos_limpios', exist_ok=True)
visitas_web_limpio.to_csv('datos_limpios/visitas_web_limpio.csv', index=False)